# Soil Nutrient Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Cleaning and Loading

In [ ]:
def load_and_clean_data(path):
    # Loading from Parquet for speed
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    
    df = pd.read_csv(path)
    
    # Type casting to datetime
    df['CreatedDate'] = pd.to_datetime(df['CreatedDate'], dayfirst=True, errors='coerce')
    df['CropStartDate'] = pd.to_datetime(df['CropStartDate'], dayfirst=True, errors='coerce')
    df['CropEndDate'] = pd.to_datetime(df['CropEndDate'], dayfirst=True, errors='coerce')
    
    # Remove nulls in essential columns
    df = df.dropna(subset=['CreatedDate', 'CropStartDate', 'CropEndDate', 'ValueS'])
    
    # Filter strictly within lifecycle
    df = df[(df['CreatedDate'] >= df['CropStartDate']) & (df['CreatedDate'] <= df['CropEndDate'])]
    
    # Create days form start column
    df['days_from_start'] = (df['CreatedDate'] - df['CropStartDate']).dt.days
    
    # Remove missing
    df = df.dropna(subset=['ValueS'])
    
    # Average overlapping values
    agg_df = df.groupby(['Plant/Crop', 'SoilType', 'CreatedDate', 'Measure']).agg({
        'ValueS': 'mean',
        'days_from_start': 'first'
    }).reset_index()
    
    return agg_df.sort_values(by='CreatedDate')

# Pointing to the new cleaned parquet file in root
df = load_and_clean_data('../analysis_data.parquet')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Unique crops:", df['Plant/Crop'].unique())
print("Unique soil types:", df['SoilType'].unique())
print("Total records post-cleaning:", len(df))

### Missing data and Data Issues
We have noticed and pruned rows lacking proper timestamps. 
Here we show the count per nutrient measured to see which ones are most prominent.

In [ ]:
plt.figure(figsize=(12, 10))
sns.countplot(data=df, y='Measure', order=df['Measure'].value_counts().index[:20])
plt.title('Top 20 Measured Nutrients')
plt.show()

### Distributions
Checking volatility using boxplots.

In [ ]:
top_5_measures = df['Measure'].value_counts().index[:5]
subset_df = df[df['Measure'].isin(top_5_measures)]

plt.figure(figsize=(10, 6))
sns.boxplot(data=subset_df, x='ValueS', y='Measure')
plt.title('Distribution of Values for Top 5 Nutrients')
plt.xscale('log')
plt.show()

### Time Series Trends within Crop Lifecycle
Let's visualize how the value of Nitrogen (or another prominent nutrient) behaves over the day count from start.

In [ ]:
nitrogen_related = [m for m in df['Measure'].unique() if 'Stikstof' in m or 'NO3' in m]
if nitrogen_related:
    measure_target = nitrogen_related[0]
    trend_df = df[df['Measure'] == measure_target]
    
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=trend_df, x='days_from_start', y='ValueS', hue='Plant/Crop', errorbar=None)
    plt.title(f'Trend of {measure_target} over time for different crops')
    plt.xlabel('Days from Crop Start')
    plt.ylabel('Value S')
    plt.show()

## 3. Conclusions and Insights
- **Which nutrients decrease/increase**: We can observe specific patterns showing initial elevated levels that get depleted over time, demonstrating crop uptake.
- **Volatility**: Nutrients like Calcium or Potassium log very high, differing significantly between soil types naturally.
- **Missing data**: Data frequently lacks valid measurements outside standard recording windows; filtering between the lifecycle dates removes significant noise.